# CareStep — Pipeline complet
**Membre : Fyras** | Projet ML avancé | Dataset UCI #296 (Diabetes 130-US Hospitals)

---

## Modèles utilisés

| # | Modèle | Algorithme | Features | Rôle |
|---|--------|-----------|----------|------|
| 1 | Régression Logistique | LogisticRegression | 12 patient | Baseline |
| 2 | LightGBM patient | LGBMClassifier | 12 patient | Modèle principal |
| 3 | LightGBM full | LGBMClassifier | 107 (toutes) | Référence |
| 4 | LightGBM calibré | Isotonic scaling | 12 patient | Probabilités fiables |
| 5 | LightGBM tuné | RandomizedSearchCV | 12 patient | Hyperparamètres optimisés |
| 6 | LightGBM tuné + calibré | Isotonic scaling | 12 patient | **Modèle final** |

**2 algorithmes** (Régression Logistique + LightGBM), **6 variantes** au total.

---

## Imports

In [ ]:
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import os
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_recall_curve, roc_curve, accuracy_score,
    classification_report, confusion_matrix
)
from scipy.stats import randint, uniform
import lightgbm as lgb
import shap

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# ── Trouver le fichier CSV automatiquement ──
# Fonctionne en local (model/) et sur Colab/Jupyter
_candidates = [
    Path('extracted_data/diabetic_data.csv'),           # si CWD = racine du projet
    Path('../extracted_data/diabetic_data.csv'),         # si CWD = model/
    Path('../../capalit_ml/diabetic_data.csv'),          # dépôt d'équipe (fyras/model/)
    Path('/content/diabetic_data.csv'),                  # Colab upload direct
    Path('/content/extracted_data/diabetic_data.csv'),   # Colab dossier extrait
]
DATA_PATH = None
for p in _candidates:
    if p.exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    # Sur Colab : uploader le CSV
    from google.colab import files
    print("⬆ Uploadez le fichier diabetic_data.csv :")
    uploaded = files.upload()
    DATA_PATH = Path(list(uploaded.keys())[0])

print(f'Fichier trouvé : {DATA_PATH}')
print('Setup OK')

---
# Lab 1 — Préparation des données

**Objectif** : Charger le dataset, nettoyer, construire les 12 features patient, et séparer train/test sans fuite.

### 1.1 Chargement du dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset : {df.shape[0]:,} encounters, {df.shape[1]} colonnes')
print(f'Patients uniques : {df["patient_nbr"].nunique():,}')
print(f'\nDistribution de readmitted :')
print(df['readmitted'].value_counts())

### 1.2 Exclusion décès / palliatifs

On retire les patients sortis en décès ou soins palliatifs — ils ne sont pas candidats à la réadmission.

In [ ]:
EXCLUDE_DISPOSITIONS = {11, 13, 14, 19, 20, 21}
mask = df['discharge_disposition_id'].isin(EXCLUDE_DISPOSITIONS)
print(f'Lignes retirées : {mask.sum():,} (dispositions {sorted(EXCLUDE_DISPOSITIONS)})')
df = df[~mask].copy()

# Protocole règle 2 : genre Unknown/Invalid retiré
gender_mask = df['gender'].isin(['Unknown/Invalid'])
print(f'Genre Unknown/Invalid retiré : {gender_mask.sum()} lignes')
df = df[~gender_mask].copy()
print(f'Restant : {df.shape[0]:,} encounters')

### 1.3 Construction de la cible binaire

$$y = 1 \text{ si readmitted} = \text{"<30"}, \text{ sinon } 0$$

In [ ]:
df['y'] = (df['readmitted'] == '<30').astype(int)
prevalence = df['y'].mean()
print(f'Positifs (réadmis <30j) : {df["y"].sum():,} / {len(df):,} ({prevalence:.1%})')
print(f'Négatifs               : {(df["y"]==0).sum():,} ({1-prevalence:.1%})')
print(f'\n→ Classe déséquilibrée : ~1 positif pour 8 négatifs')

### 1.4 Construction des 12 features patient

On n'utilise que des features qu'un patient peut répondre lui-même à la sortie.

In [ ]:
# 1. Âge — ordinal (0 à 9)
AGE_ORDER = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
             '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
df['age_ord'] = df['age'].map({a: i for i, a in enumerate(AGE_ORDER)})

# 2. Genre — binaire
df['gender_bin'] = (df['gender'] == 'Male').astype(int)

# 3-4. Hospitalisations et urgences passées — cap à 3
df['inpatient_cap'] = df['number_inpatient'].clip(upper=3)
df['emergency_cap'] = df['number_emergency'].clip(upper=3)

# 5. Consultations passées — buckets
df['outpatient_bkt'] = df['number_outpatient'].apply(
    lambda x: 0 if x == 0 else (1 if x <= 2 else (2 if x <= 5 else 3)))

# 6. time_in_hospital — tel quel (1-14)

# 7. Diagnostic principal — regroupement ICD-9 en 7 familles
def group_diag(code):
    if pd.isna(code) or code == '?':
        return 'other'
    try:
        num = float(code)
    except ValueError:
        return 'injury' if str(code).startswith('E') else 'other'
    if 390 <= num <= 459 or num == 785: return 'circulatory'
    if 460 <= num <= 519 or num == 786: return 'respiratory'
    if 250 <= num < 251: return 'diabetes'
    if 520 <= num <= 579 or num == 787: return 'digestive'
    if 580 <= num <= 629 or 1 <= num <= 139 or 680 <= num <= 709 or num == 788:
        return 'infection_renal'
    if 800 <= num <= 999: return 'injury'
    return 'other'

df['diag_group'] = df['diag_1'].apply(group_diag)

# 8. num_medications — tel quel

# 9. insulin — déjà catégoriel (No, Steady, Up, Down)

# 10. change — déjà Ch / No

# 11. A1C testé — binaire
# Attention : pandas lit le texte 'None' du CSV comme NaN → NaN = test non fait
df['a1c_tested'] = df['A1Cresult'].notna().astype(int)

# 12. Destination de sortie — 3 classes
df['discharge_group'] = df['discharge_disposition_id'].apply(
    lambda d: 'home' if d in (1, 8) else ('home_with_help' if d == 6 else 'facility'))

print('12 features patient construites ✓')
print(f'\nDistribution diag_group :')
print(df['diag_group'].value_counts().to_string())

### 1.5 Split anti-fuite par patient

Un patient n'apparaît que dans train **OU** test — jamais les deux. Cela évite que le modèle "reconnaisse" un patient déjà vu.

In [ ]:
NUM_FEATURES = ['age_ord', 'gender_bin', 'inpatient_cap', 'emergency_cap',
                'outpatient_bkt', 'time_in_hospital', 'num_medications', 'a1c_tested']
CAT_FEATURES = ['diag_group', 'insulin', 'change', 'discharge_group']

# Features complètes (pour le modèle de référence)
FULL_NUM = NUM_FEATURES + ['num_lab_procedures', 'num_procedures',
                           'number_diagnoses', 'admission_type_id', 'admission_source_id']
FULL_CAT = CAT_FEATURES + ['medical_specialty']   # race : audit d'équité uniquement, jamais en entrée
df['medical_specialty'] = df['medical_specialty'].replace('?', 'Unknown')
df['race'] = df['race'].replace('?', 'Unknown')

# Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, df['y'], df['patient_nbr'].values))
df_train, df_test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

# Vérification
overlap = set(df_train['patient_nbr']) & set(df_test['patient_nbr'])
print(f'Train : {len(df_train):,} encounters ({df_train["patient_nbr"].nunique():,} patients)')
print(f'Test  : {len(df_test):,} encounters ({df_test["patient_nbr"].nunique():,} patients)')
print(f'Patients en commun : {len(overlap)} (doit être 0) ✓')

### 1.6 Preprocessing sklearn

In [ ]:
# Preprocesseur 12 features patient
patient_prep = ColumnTransformer([
    ('num', 'passthrough', NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='infrequent_if_exist',
                          sparse_output=False, drop='if_binary'), CAT_FEATURES),
], remainder='drop', verbose_feature_names_out=False)

# Preprocesseur toutes features (référence)
full_prep = ColumnTransformer([
    ('num', 'passthrough', FULL_NUM),
    ('cat', OneHotEncoder(handle_unknown='infrequent_if_exist',
                          sparse_output=False, drop='if_binary'), FULL_CAT),
], remainder='drop', verbose_feature_names_out=False)

X_train_p = patient_prep.fit_transform(df_train)
X_test_p = patient_prep.transform(df_test)
X_train_f = full_prep.fit_transform(df_train)
X_test_f = full_prep.transform(df_test)

y_train = df_train['y'].values
y_test = df_test['y'].values
groups = df_train['patient_nbr'].values
patient_names = list(patient_prep.get_feature_names_out())
full_names = list(full_prep.get_feature_names_out())

pos_rate = y_train.mean()
scale_pos = (1 - pos_rate) / pos_rate

print(f'Features patient : {X_train_p.shape[1]} colonnes après one-hot')
print(f'Features full    : {X_train_f.shape[1]} colonnes après one-hot')
print(f'Prévalence train : {pos_rate:.3f}')
print(f'scale_pos_weight : {scale_pos:.1f}')
print(f'\n✓ Lab 1 terminé — données prêtes')

---
# Lab 2 — Modélisation

**3 étapes progressives** :
1. Régression logistique → baseline simple
2. LightGBM 12 features → modèle principal
3. LightGBM toutes features → référence (combien on perd ?)

Puis calibration pour que les probabilités soient fiables.

### Modèle 1 — Régression Logistique (baseline)

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000,
                        solver='lbfgs', random_state=42)
lr.fit(X_train_p, y_train)

lr_proba = lr.predict_proba(X_test_p)[:, 1]

print('Modèle 1 — Régression Logistique')
print(f'  Algorithme : LogisticRegression (class_weight=balanced)')
print(f'  Features   : 12 patient ({X_train_p.shape[1]} après one-hot)')
print(f'  PR-AUC     : {average_precision_score(y_test, lr_proba):.4f}')
print(f'  ROC-AUC    : {roc_auc_score(y_test, lr_proba):.4f}')
print(f'  Brier      : {brier_score_loss(y_test, lr_proba):.4f}')

print(f'\nCoefficients les plus importants :')
coefs = sorted(zip(patient_names, lr.coef_[0]), key=lambda x: -abs(x[1]))
for name, c in coefs[:5]:
    print(f'    {name:25s} : {c:+.3f}')

### Modèle 2 — LightGBM patient (12 features)

In [ ]:
lgbm_patient = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=5, num_leaves=31,
    min_child_samples=50, scale_pos_weight=scale_pos,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbose=-1, n_jobs=-1,
)
lgbm_patient.fit(X_train_p, y_train)

lgbm_p_proba = lgbm_patient.predict_proba(X_test_p)[:, 1]

print('Modèle 2 — LightGBM patient')
print(f'  Algorithme : LGBMClassifier (scale_pos_weight={scale_pos:.1f})')
print(f'  Features   : 12 patient ({X_train_p.shape[1]} après one-hot)')
print(f'  PR-AUC     : {average_precision_score(y_test, lgbm_p_proba):.4f}')
print(f'  ROC-AUC    : {roc_auc_score(y_test, lgbm_p_proba):.4f}')
print(f'  Brier      : {brier_score_loss(y_test, lgbm_p_proba):.4f}')

print(f'\nFeature importance (top 5) :')
fi = sorted(zip(patient_names, lgbm_patient.feature_importances_), key=lambda x: -x[1])
for name, imp in fi[:5]:
    print(f'    {name:25s} : {imp}')

### Modèle 3 — LightGBM full (toutes features — référence)

In [ ]:
lgbm_full = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6, num_leaves=63,
    min_child_samples=50, scale_pos_weight=scale_pos,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbose=-1, n_jobs=-1,
)
lgbm_full.fit(X_train_f, y_train)

lgbm_f_proba = lgbm_full.predict_proba(X_test_f)[:, 1]

print('Modèle 3 — LightGBM FULL (référence)')
print(f'  Algorithme : LGBMClassifier')
print(f'  Features   : toutes ({X_train_f.shape[1]} après one-hot)')
print(f'  PR-AUC     : {average_precision_score(y_test, lgbm_f_proba):.4f}')
print(f'  ROC-AUC    : {roc_auc_score(y_test, lgbm_f_proba):.4f}')
print(f'  Brier      : {brier_score_loss(y_test, lgbm_f_proba):.4f}')

# Ratio patient / full
ratio_pf = average_precision_score(y_test, lgbm_p_proba) / average_precision_score(y_test, lgbm_f_proba)
print(f'\n→ Ratio PR-AUC patient/full = {ratio_pf:.1%} (cible ≥ 85-90%)')

### Modèle 4 — LightGBM calibré (isotonic scaling)

Le LightGBM produit des **scores**, pas des **probabilités fiables**. La calibration isotonique corrige ça.

In [ ]:
# Séparer train en 80% entraînement + 20% calibration (par patient)
cal_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=99)
cal_train_idx, cal_val_idx = next(cal_split.split(X_train_p, y_train, groups))

# Entraîner un LightGBM frais sur 80%
lgbm_for_cal = lgb.LGBMClassifier(**lgbm_patient.get_params())
lgbm_for_cal.fit(X_train_p[cal_train_idx], y_train[cal_train_idx])

# Calibrer sur les 20% restants
cal_model = CalibratedClassifierCV(lgbm_for_cal, method='isotonic', cv='prefit')
cal_model.fit(X_train_p[cal_val_idx], y_train[cal_val_idx])

cal_proba = cal_model.predict_proba(X_test_p)[:, 1]

print('Modèle 4 — LightGBM calibré')
print(f'  Base       : LightGBM patient (12 features)')
print(f'  Méthode    : Calibration isotonique (prefit)')
print(f'  PR-AUC     : {average_precision_score(y_test, cal_proba):.4f}')
print(f'  ROC-AUC    : {roc_auc_score(y_test, cal_proba):.4f}')
print(f'  Brier      : {brier_score_loss(y_test, cal_proba):.4f}')
print(f'\n  Moyenne des probas calibrées : {cal_proba.mean():.4f} (prévalence réelle : {y_test.mean():.4f})')

---
# Lab 3 — Évaluation

Comparaison des 4 premiers modèles + SHAP pour l'explicabilité.

### 3.1 Tableau comparatif des métriques

In [ ]:
results = {}
for name, proba in [
    ('1. LogReg (12f)', lr_proba),
    ('2. LightGBM (12f)', lgbm_p_proba),
    ('3. LightGBM FULL', lgbm_f_proba),
    ('4. LightGBM calibré', cal_proba),
]:
    results[name] = {
        'PR-AUC': average_precision_score(y_test, proba),
        'ROC-AUC': roc_auc_score(y_test, proba),
        'Brier': brier_score_loss(y_test, proba),
    }

metrics_df = pd.DataFrame(results).T
print(metrics_df.round(4).to_string())
print(f'\nRatio patient/full PR-AUC = {ratio_pf:.1%}')

### 3.2 Niveaux de risque

In [ ]:
threshold_med = pos_rate       # ~11%
threshold_high = 2 * pos_rate  # ~22%

risk = np.where(cal_proba >= threshold_high, 'high',
                np.where(cal_proba >= threshold_med, 'medium', 'low'))

total_re = y_test.sum()
print(f'Seuils : medium ≥ {threshold_med:.3f}, high ≥ {threshold_high:.3f}')
print()
for level in ['low', 'medium', 'high']:
    m = risk == level
    n = m.sum()
    tp = y_test[m].sum()
    rate = y_test[m].mean()
    capture = tp / total_re
    print(f'  {level:6s} : {n:5,} patients ({n/len(y_test):.1%}), '
          f'taux réadm = {rate:.1%}, capture = {capture:.1%}')

### 3.3 Courbes PR et ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR curve
for name, proba, ls in [
    ('LogReg', lr_proba, '--'),
    ('LightGBM patient', lgbm_p_proba, '-'),
    ('LightGBM calibré', cal_proba, '-'),
    ('LightGBM FULL', lgbm_f_proba, ':'),
]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[0].plot(rec, prec, ls, label=f'{name} (AP={ap:.3f})')
axes[0].axhline(y_test.mean(), color='gray', ls=':', label=f'Baseline ({y_test.mean():.3f})')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Courbe Precision-Recall')
axes[0].legend(fontsize=8)

# ROC curve
for name, proba, ls in [
    ('LogReg', lr_proba, '--'),
    ('LightGBM patient', lgbm_p_proba, '-'),
    ('LightGBM calibré', cal_proba, '-'),
    ('LightGBM FULL', lgbm_f_proba, ':'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[1].plot(fpr, tpr, ls, label=f'{name} (AUC={auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k:', alpha=0.3)
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('Courbe ROC')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 3.4 Courbe de calibration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, proba, ls in [('LightGBM brut', lgbm_p_proba, '--'), ('LightGBM calibré', cal_proba, '-')]:
    prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10, strategy='uniform')
    axes[0].plot(prob_pred, prob_true, ls, marker='o', label=name)
axes[0].plot([0, 1], [0, 1], 'k:', alpha=0.3, label='Parfaitement calibré')
axes[0].set_xlabel('Probabilité prédite')
axes[0].set_ylabel('Fraction observée')
axes[0].set_title('Courbe de calibration')
axes[0].legend()

axes[1].hist(cal_proba[y_test == 0], bins=50, alpha=0.6, label='Non réadmis', density=True)
axes[1].hist(cal_proba[y_test == 1], bins=50, alpha=0.6, label='Réadmis <30j', density=True)
axes[1].axvline(threshold_med, color='orange', ls='--', label=f'Seuil medium ({threshold_med:.2f})')
axes[1].axvline(threshold_high, color='red', ls='--', label=f'Seuil high ({threshold_high:.2f})')
axes[1].set_xlabel('Probabilité calibrée')
axes[1].set_ylabel('Densité')
axes[1].set_title('Distribution des probabilités')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 3.5 SHAP — Explicabilité (ODS-B)

SHAP décompose chaque prédiction en contributions par feature. On identifie les **facteurs modifiables** — ceux sur lesquels le patient peut agir.

In [ ]:
explainer = shap.TreeExplainer(lgbm_patient)
shap_idx = np.random.RandomState(42).choice(len(X_test_p), size=2000, replace=False)
X_shap = X_test_p[shap_idx]
shap_values = explainer.shap_values(X_shap)
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

shap.summary_plot(shap_vals, X_shap, feature_names=patient_names,
                  max_display=len(patient_names), show=True)

In [ ]:
shap.summary_plot(shap_vals, X_shap, feature_names=patient_names,
                  plot_type='bar', max_display=len(patient_names), show=True)

### 3.6 Facteurs modifiables → Actions

In [ ]:
MODIFIABLE = {
    'insulin':         'Prenez votre nouvelle dose d\'insuline comme prescrit',
    'change':          'Suivez les changements de médicaments prescrits à la sortie',
    'num_medications': 'Utilisez un pilulier pour organiser vos médicaments',
    'a1c_tested':      'Planifiez un test HbA1c avec votre médecin',
    'discharge_group': 'Respectez le plan de suivi de votre lieu de sortie',
    'outpatient_bkt':  'Prenez un rendez-vous de suivi dans les 7 jours',
    'emergency_cap':   'En cas de symptômes, appelez votre médecin avant d\'aller aux urgences',
}

def get_base(fname):
    for base in MODIFIABLE:
        if fname == base or fname.startswith(base + '_'):
            return base
    return None

mean_shap = np.abs(shap_vals).mean(axis=0)
for fname, imp in sorted(zip(patient_names, mean_shap), key=lambda x: -x[1]):
    base = get_base(fname)
    tag = '✓ MODIFIABLE' if base else '  non-modifiable'
    print(f'  {fname:28s}  SHAP={imp:.4f}  {tag}')
    if base:
        print(f'    → {MODIFIABLE[base]}')

---
# Lab 4 — Tuning

**Objectif** : Optimiser les hyperparamètres du LightGBM patient avec RandomizedSearchCV.

### 4.1 Recherche d'hyperparamètres

In [ ]:
param_distributions = {
    'n_estimators':      randint(200, 800),
    'learning_rate':     uniform(0.01, 0.14),
    'max_depth':         randint(3, 8),
    'num_leaves':        randint(15, 63),
    'min_child_samples': randint(20, 100),
    'subsample':         uniform(0.6, 0.35),
    'colsample_bytree':  uniform(0.5, 0.45),
    'reg_alpha':         uniform(0, 1.0),
    'reg_lambda':        uniform(0.5, 2.5),
}

base = lgb.LGBMClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=-1, n_jobs=-1)
gkf = GroupKFold(n_splits=5)

search = RandomizedSearchCV(
    base, param_distributions, n_iter=40, scoring='average_precision',
    cv=gkf, random_state=42, verbose=1, n_jobs=-1,
)

print('Recherche : 40 combinaisons × 5 folds = 200 ajustements')
print('Métrique  : average_precision (PR-AUC)\n')
search.fit(X_train_p, y_train, groups=groups)

print(f'\nMeilleur score CV : {search.best_score_:.4f}')

### 4.2 Meilleurs hyperparamètres

In [ ]:
print('Modèle 5 — LightGBM tuné')
print('\nMeilleurs hyperparamètres :')
for p, v in sorted(search.best_params_.items()):
    print(f'  {p:22s} = {v:.4f}' if isinstance(v, float) else f'  {p:22s} = {v}')

### 4.3 Comparaison avant / après tuning

In [ ]:
best_model = search.best_estimator_
tuned_proba = best_model.predict_proba(X_test_p)[:, 1]

before_pr = average_precision_score(y_test, lgbm_p_proba)
after_pr = average_precision_score(y_test, tuned_proba)
before_roc = roc_auc_score(y_test, lgbm_p_proba)
after_roc = roc_auc_score(y_test, tuned_proba)

print(f'         {"Avant":>10s} {"Après":>10s} {"Δ":>10s}')
print(f'  PR-AUC  {before_pr:10.4f} {after_pr:10.4f} {(after_pr-before_pr)/before_pr*100:+9.1f}%')
print(f'  ROC-AUC {before_roc:10.4f} {after_roc:10.4f} {(after_roc-before_roc)/before_roc*100:+9.1f}%')

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(2)
ax.bar(x - 0.15, [before_pr, before_roc], 0.3, label='Avant tuning', color='#f0a500')
ax.bar(x + 0.15, [after_pr, after_roc], 0.3, label='Après tuning', color='#4ecca3')
ax.set_xticks(x)
ax.set_xticklabels(['PR-AUC', 'ROC-AUC'])
ax.set_title('Impact du tuning')
ax.legend()
plt.tight_layout()
plt.show()

### Modèle 6 — LightGBM tuné + calibré (modèle final)

In [ ]:
tuned_for_cal = lgb.LGBMClassifier(**best_model.get_params())
tuned_for_cal.fit(X_train_p[cal_train_idx], y_train[cal_train_idx])
cal_tuned = CalibratedClassifierCV(tuned_for_cal, method='isotonic', cv='prefit')
cal_tuned.fit(X_train_p[cal_val_idx], y_train[cal_val_idx])

final_proba = cal_tuned.predict_proba(X_test_p)[:, 1]
final_pr = average_precision_score(y_test, final_proba)
final_roc = roc_auc_score(y_test, final_proba)
final_brier = brier_score_loss(y_test, final_proba)

print('Modèle 6 — LightGBM tuné + calibré (MODÈLE FINAL)')
print(f'  PR-AUC  : {final_pr:.4f}')
print(f'  ROC-AUC : {final_roc:.4f}')
print(f'  Brier   : {final_brier:.4f}')

### 4.4 Niveaux de risque (modèle final)

In [ ]:
risk_final = np.where(final_proba >= threshold_high, 'high',
                      np.where(final_proba >= threshold_med, 'medium', 'low'))

print(f'Seuils : medium ≥ {threshold_med:.3f}, high ≥ {threshold_high:.3f}\n')
for level in ['low', 'medium', 'high']:
    m = risk_final == level
    n = m.sum()
    tp = y_test[m].sum()
    rate = y_test[m].mean()
    capture = tp / total_re
    print(f'  {level:6s} : {n:5,} patients ({n/len(y_test):.1%}), '
          f'taux réadm = {rate:.1%}, capture = {capture:.1%}')

mh_capture = y_test[risk_final != 'low'].sum() / total_re
print(f'\n  MEDIUM + HIGH capturent {mh_capture:.1%} des réadmissions')

---
# Résumé final — Tous les modèles

In [ ]:
all_results = {
    '1. LogReg (12f)':           {'PR-AUC': average_precision_score(y_test, lr_proba),
                                  'ROC-AUC': roc_auc_score(y_test, lr_proba),
                                  'Brier': brier_score_loss(y_test, lr_proba)},
    '2. LightGBM (12f)':         {'PR-AUC': average_precision_score(y_test, lgbm_p_proba),
                                  'ROC-AUC': roc_auc_score(y_test, lgbm_p_proba),
                                  'Brier': brier_score_loss(y_test, lgbm_p_proba)},
    '3. LightGBM FULL':          {'PR-AUC': average_precision_score(y_test, lgbm_f_proba),
                                  'ROC-AUC': roc_auc_score(y_test, lgbm_f_proba),
                                  'Brier': brier_score_loss(y_test, lgbm_f_proba)},
    '4. LightGBM calibré':       {'PR-AUC': average_precision_score(y_test, cal_proba),
                                  'ROC-AUC': roc_auc_score(y_test, cal_proba),
                                  'Brier': brier_score_loss(y_test, cal_proba)},
    '5. LightGBM tuné':          {'PR-AUC': after_pr,
                                  'ROC-AUC': after_roc,
                                  'Brier': brier_score_loss(y_test, tuned_proba)},
    '6. Tuné+calibré (FINAL)':   {'PR-AUC': final_pr,
                                  'ROC-AUC': final_roc,
                                  'Brier': final_brier},
}

summary_df = pd.DataFrame(all_results).T
print('TABLEAU RÉCAPITULATIF — 6 MODÈLES')
print('=' * 55)
print(summary_df.round(4).to_string())
print(f'\nAlgorithmes : Régression Logistique + LightGBM')
print(f'Modèle final : #6 (LightGBM tuné + calibré)')

In [ ]:
# Graphique récapitulatif
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

names = list(all_results.keys())
short = ['LogReg', 'LGBM', 'FULL', 'Calibré', 'Tuné', 'Final']

# PR-AUC
vals = [all_results[n]['PR-AUC'] for n in names]
colors = ['#4da8da'] * 5 + ['#4ecca3']
axes[0].bar(short, vals, color=colors)
axes[0].axhline(y_test.mean(), color='red', ls=':', label=f'Baseline ({y_test.mean():.3f})')
axes[0].set_title('PR-AUC par modèle')
axes[0].set_ylim(0, 0.25)
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=30)

# ROC-AUC
vals = [all_results[n]['ROC-AUC'] for n in names]
axes[1].bar(short, vals, color=colors)
axes[1].axhline(0.5, color='red', ls=':', label='Aléatoire (0.5)')
axes[1].set_title('ROC-AUC par modèle')
axes[1].set_ylim(0.4, 0.75)
axes[1].legend(fontsize=8)
axes[1].tick_params(axis='x', rotation=30)

# Brier
vals = [all_results[n]['Brier'] for n in names]
axes[2].bar(short, vals, color=colors)
axes[2].set_title('Brier score (↓ = mieux)')
axes[2].set_ylim(0, 0.25)
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('CareStep — Comparaison des 6 modèles', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Pourquoi pas l'accuracy ?

Avec 89% de négatifs, un modèle qui dit toujours "non" obtient **89% d'accuracy**. C'est pourquoi on utilise **PR-AUC** (adapté aux classes déséquilibrées) et **Brier** (calibration des probabilités).

In [ ]:
print('Accuracy (pour information — métrique trompeuse ici) :')
print(f'  Modèle naïf (tout = 0)  : {1 - y_test.mean():.1%}')
print(f'  Modèle final (seuil 0.5): {accuracy_score(y_test, (final_proba >= 0.5).astype(int)):.1%}')
print(f'\n→ Les deux donnent ~89% — l\'accuracy ne distingue pas un bon modèle d\'un modèle inutile.')
print(f'→ Notre modèle est utile grâce à la stratification : HIGH = {y_test[risk_final=="high"].mean():.1%} vs LOW = {y_test[risk_final=="low"].mean():.1%}')